# ETL - Preparación de datos para modelado

**Objetivo:** Transformar los datos limpios (después del EDA) en un conjunto listo para entrenar modelos de machine learning.

**Pasos a realizar:**
1. Aplicar transformación logarítmica (`log1p`) a variables con sesgo positivo (`horas_formacion_mes`, `distancia_oficina_km`, `bono_anual`).
2. Codificar variables categóricas: binarias a 0/1, multicategóricas con one‑hot encoding.
3. Separar predictores (X) y objetivo (y).
4. Dividir en entrenamiento (80%) y prueba (20%).
5. Estandarizar (escalar) las variables numéricas (media=0, desviación=1).
6. Guardar los conjuntos procesados en `data/processed/` y el escalador en `models/`.

**Nota:** Este notebook asume que ya se ha ejecutado el EDA (`01_eda.ipynb`) y que los datos originales están en `data/raw/productividad_empleados.csv`.

In [8]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Cargar datos originales
df = pd.read_csv('../data/raw/productividad_empleados.csv')

# Verificar que se cargó bien
print("Shape:", df.shape)   
print("Primeras filas:")
df.head()

Shape: (2500, 12)
Primeras filas:


,productividad,edad,sexo,hijos,años_empresa,satisfaccion_equipo,horas_formacion_mes,distancia_oficina_km,departamento,estres_reportado,bono_anual,jornada_continua
0,26,60,Mujer,No,17,4,4.1,24.1,IT,5,4.8,No
1,25,50,Hombre,No,1,3,7.8,2.1,Marketing,2,6.3,No
2,21,36,Mujer,Sí,29,2,12.1,2.6,Operaciones,2,3.2,No
3,29,64,Hombre,No,1,7,12.3,0.9,Marketing,1,2.9,No
4,29,29,Hombre,Sí,9,9,5.9,1.1,RRHH,3,3.0,No


In [9]:
# Aplicar transformación log1p a variables con sesgo positivo
columnas_sesgo = ['horas_formacion_mes', 'distancia_oficina_km', 'bono_anual']

for col in columnas_sesgo:
    # Crear nueva columna con el logaritmo natural de (valor+1)
    df[col + '_log'] = np.log1p(df[col])
    
    # Calcular skewness (asimetría) antes y después para ver reducción de sesgo
    sesgo_original = df[col].skew()
    sesgo_nuevo = df[col + '_log'].skew()
    print(f"{col}: skew original = {sesgo_original:.2f}, skew transformado = {sesgo_nuevo:.2f}")

# Mostrar primeras filas para verificar nuevas columnas
df.head()

horas_formacion_mes: skew original = 1.29, skew transformado = -0.14
distancia_oficina_km: skew original = 1.92, skew transformado = 0.11
bono_anual: skew original = 1.24, skew transformado = -0.06


,productividad,edad,sexo,hijos,años_empresa,satisfaccion_equipo,horas_formacion_mes,distancia_oficina_km,departamento,estres_reportado,bono_anual,jornada_continua,horas_formacion_mes_log,distancia_oficina_km_log,bono_anual_log
0,26,60,Mujer,No,17,4,4.1,24.1,IT,5,4.8,No,1.629241,3.222868,1.757858
1,25,50,Hombre,No,1,3,7.8,2.1,Marketing,2,6.3,No,2.174752,1.131402,1.987874
2,21,36,Mujer,Sí,29,2,12.1,2.6,Operaciones,2,3.2,No,2.572612,1.280934,1.435085
3,29,64,Hombre,No,1,7,12.3,0.9,Marketing,1,2.9,No,2.587764,0.641854,1.360977
4,29,29,Hombre,Sí,9,9,5.9,1.1,RRHH,3,3.0,No,1.931521,0.741937,1.386294


## Transformación logarítmica (log1p) para reducir sesgo

Se aplicó `log1p` a las variables `horas_formacion_mes`, `distancia_oficina_km` y `bono_anual` para corregir su sesgo positivo.

**Resultado:**
- Antes: sesgos entre 1.24 y 1.92 (asimetría fuerte).
- Después: sesgos entre -0.14 y 0.11 (prácticamente simétricos).

La transformación fue efectiva. Las nuevas columnas `_log` se usarán en el modelo en lugar de las originales.

In [10]:
# Codificación de variables categóricas

# Variables binarias (dos categorías): mapear a 0 y 1
mapeo_hijos = {'No': 0, 'Sí': 1}
df['hijos_bin'] = df['hijos'].map(mapeo_hijos)

mapeo_jornada = {'No': 0, 'Sí': 1}
df['jornada_bin'] = df['jornada_continua'].map(mapeo_jornada)

# Variables multicategóricas (varias categorías): usar one-hot encoding
# Esto crea columnas dummy: por ejemplo, sexo_Mujer, sexo_Hombre, etc.
df = pd.get_dummies(df, columns=['sexo', 'departamento'], drop_first=True)
# drop_first=True evita multicolinealidad (elimina una categoría de referencia)

# Verificar las nuevas columnas
print("Columnas después de codificar:")
print([col for col in df.columns if 'sexo_' in col or 'departamento_' in col or col.endswith('_bin')])

Columnas después de codificar:
['hijos_bin', 'jornada_bin', 'sexo_Mujer', 'sexo_No binario', 'departamento_Marketing', 'departamento_Operaciones', 'departamento_RRHH', 'departamento_Ventas']


## Codificación de variables categóricas

Se transformaron las variables categóricas a formato numérico:

- `hijos`: mapeo Sí→1, No→0 (nueva columna `hijos_bin`).
- `jornada_continua`: mapeo Sí→1, No→0 (nueva columna `jornada_bin`).
- `sexo` y `departamento`: one‑hot encoding con `drop_first=True` para evitar multicolinealidad. Se crearon las siguientes columnas:
  - `sexo_Mujer`, `sexo_No binario` (la categoría `Hombre` queda como referencia).
  - `departamento_Marketing`, `departamento_Operaciones`, `departamento_RRHH`, `departamento_Ventas` (la categoría `IT` queda como referencia).

Las columnas originales (`hijos`, `jornada_continua`, `sexo`, `departamento`) fueron eliminadas. El DataFrame ahora contiene las versiones numéricas listas para el modelo.

In [11]:
# Separar los datos en variables predictoras (X) y variable objetivo (y)

# Lista de las columnas que usaremos para predecir la productividad.
# Incluye las variables numéricas originales (sin sesgo), las versiones transformadas con logaritmo,
# las variables binarias (hijos_bin, jornada_bin) y las columnas dummy creadas con one-hot encoding.
columnas_predictoras = [
    'edad', 'años_empresa', 'satisfaccion_equipo', 'estres_reportado',
    'horas_formacion_mes_log', 'distancia_oficina_km_log', 'bono_anual_log',
    'hijos_bin', 'jornada_bin',
    'sexo_Mujer', 'sexo_No binario',
    'departamento_Marketing', 'departamento_Operaciones', 'departamento_RRHH', 'departamento_Ventas'
]

# Verificar que todas las columnas existen en el DataFrame (por si algún nombre cambió)
faltan = [col for col in columnas_predictoras if col not in df.columns]
if faltan:
    print("Faltan estas columnas:", faltan)
else:
    # Si todo está bien, asignamos X (predictoras) y y (objetivo)
    X = df[columnas_predictoras]
    y = df['productividad']
    print("Forma de X:", X.shape)
    print("Forma de y:", y.shape)

Forma de X: (2500, 15)
Forma de y: (2500,)


## Separación de predictoras (X) y objetivo (y)

- `X` contiene 2500 observaciones y 15 variables predictoras.
- `y` contiene la variable objetivo `productividad` (2500 valores).

Listo para la división en entrenamiento y prueba.

In [12]:
# Se dividen los datos en entrenamiento (80%) y prueba (20%)
# Usamos una semilla (random_state) para que la división sea reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

# Verificar las dimensiones resultantes
print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)    
print("Tamaño de y_train:", y_train.shape)
print("Tamaño de y_test:", y_test.shape)

Tamaño de X_train: (2000, 15)
Tamaño de X_test: (500, 15)
Tamaño de y_train: (2000,)
Tamaño de y_test: (500,)


In [13]:
# Estandarizar variables numéricas continuas (media 0, desviación 1)


# Columnas que se van a escalar (solo las continuas)
columnas_a_escalar = [
    'edad', 'años_empresa', 'satisfaccion_equipo', 'estres_reportado',
    'horas_formacion_mes_log', 'distancia_oficina_km_log', 'bono_anual_log'
]

# Copiamos los conjuntos para no modificar los originales
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Crear el escalador
scaler = StandardScaler()

# Ajustar (calcular media y desviación) en entrenamiento y transformar
X_train_scaled[columnas_a_escalar] = scaler.fit_transform(X_train[columnas_a_escalar])

# Aplicar la misma transformación a prueba (sin recalcular)
X_test_scaled[columnas_a_escalar] = scaler.transform(X_test[columnas_a_escalar])

# Verificación rápida
print("Media de 'edad' escalada (train):", X_train_scaled['edad'].mean())
print("Desviación estándar:", X_train_scaled['edad'].std())

Media de 'edad' escalada (train): -4.9737991503207014e-17
Desviación estándar: 1.0002500937890797


In [14]:
# Guardar los conjuntos escalados y el escalador

# Crear las carpetas si no existen
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Guardar DataFrames en CSV
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Guardar el escalador
joblib.dump(scaler, '../models/scaler.pkl')

print("Datos guardados en data/processed/")
print("Escalador guardado en models/scaler.pkl")

Datos guardados en data/processed/
Escalador guardado en models/scaler.pkl


## Fin del ETL

Los datos han sido transformados, codificados, escalados y guardados en `data/processed/`.  
El escalador está en `models/scaler.pkl`.  
Estamos listos para pasar al notebook de modelado (`03_modelado.ipynb`).